---------

Comments
- 9% of location data are missing, but:
    - For active users we do not need location data
    - The missing data will average out (likely the 9% is MCAR)

--------
# Barter Deals dataset


- Construct the main predictor $apps\_after\_7\_days$
- Construct MAU/WAU (monthly and weekly active users, based on how many creators applied to a deal)
- Use location data to get estimates on the nr. of 'eligible' creators: the number of creators that are withina certain range of a (physical) deal

In [14]:
import numpy as np
import pandas as pd
from src import paths
from collections import defaultdict


In [51]:
df_apps = pd.read_parquet(paths.RAW_DATA_DIR / 'BARTER_DEAL_APPLICATIONS.parquet')
df_deals = pd.read_parquet(paths.PROCESSED_DATA_DIR / 'BARTER_DEALS_CLEAN.parquet')

df_deals = pd.merge(df_deals, df_apps[['deal_id', 'company_location_id']].drop_duplicates(subset=['deal_id']), on='deal_id', how='left')

# Generate metrics

## Nr of active creators 


In [16]:
def get_daily_uniques(target_dates_ns, event_dates, creators, window_days):
    window_ns = pd.Timedelta(days=window_days).value
    n_events = len(event_dates)
    results = []
    
    counts = defaultdict(int)
    unique_count = 0
    
    left_ptr = 0
    right_ptr = 0
    
    # Iterate through each calendar day
    for current_day_ns in target_dates_ns:
        min_time_ns = current_day_ns - window_ns
        
        # EXPAND: Add events that happened before the current day
        while right_ptr < n_events and event_dates[right_ptr] < current_day_ns:
            inf = creators[right_ptr]
            if counts[inf] == 0:
                unique_count += 1
            counts[inf] += 1
            right_ptr += 1
            
        # SHRINK: Remove events that happened on or before the min_time (falling out of window)
        while left_ptr < right_ptr and event_dates[left_ptr] <= min_time_ns:
            inf = creators[left_ptr]
            counts[inf] -= 1
            if counts[inf] == 0:
                unique_count -= 1
            left_ptr += 1
            
        # Record the snapshot for this specific day
        results.append(unique_count)
        
    return results


In [ ]:
df_apps_sorted = df_apps.sort_values('application_created_at').reset_index(drop=True)


event_dates = df_apps_sorted['application_created_at'].astype('int64').values
dateranges = pd.date_range(
    start=df_apps_sorted['application_created_at'].min().normalize(), 
    end=df_apps_sorted['application_created_at'].max().normalize(), 
    freq='D'
)
target_dates_ns = dateranges.astype('int64').values


creators = df_apps['creator_id'].values
# Convert the target daily dates to nanoseconds
df_active_creators = pd.DataFrame({'date': dateranges})
df_active_creators['active_last_month'] = get_daily_uniques(
    target_dates_ns, event_dates, creators, window_days=30
)
df_active_creators['active_last_week'] = get_daily_uniques(
    target_dates_ns, event_dates, creators, window_days=7
)

## Apps after 7 days

In [19]:
# # Known to be correct (ground truth), but inefficient; prefer vectorized function (faster)

# def apps_after_n_days(row, n=7):
#     subs = df_apps[df_apps['deal_id'] == row['deal_id']]
#     if len(subs) != row['applicants_applications_count']:
#         subs = subs[subs['deleted_at'].isna()]

#     # "created_at" is the date on which the application log entry was created (application was made)
#     days_since_live = (subs.application_created_at - row.live_since).dt.days
#     return ((days_since_live >= 0) & (days_since_live <= n)).sum()

# apps_after_7_days = df_deals.apply(apps_after_n_days, axis=1)
# df_deals['apps_after_7_days'] = apps_after_7_days


def calculate_target_vectorized(df_deals, df_apps, n=7):
    # 1. Pre-calculate the total rows in df_apps per deal to replicate your exact 'quirk' logic
    app_counts = df_apps.groupby('deal_id').size().reset_index(name='actual_app_count')
    
    # 2. Merge only the necessary columns to save memory
    merged = df_apps[['deal_id', 'application_created_at', 'deleted_at']].merge(
        df_deals[['deal_id', 'live_since', 'applicants_applications_count']], 
        on='deal_id', 
        how='inner'
    )
    
    # Add our actual counts back into the merged dataframe
    merged = merged.merge(app_counts, on='deal_id', how='left')

    # 3. Calculate the days since live in one vectorized sweep
    # (Ensure both columns are datetime objects before this step)
    merged['days_since_live'] = (merged['application_created_at'] - merged['live_since']).dt.days

    # 4. Apply the time window mask
    time_mask = (merged['days_since_live'] >= 0) & (merged['days_since_live'] <= n)
    
    # 5. Apply your specific data-quirk mask (deleted_at logic)
    quirk_mask = (merged['actual_app_count'] == merged['applicants_applications_count']) | \
                 (merged['deleted_at'].isna())
                 
    # Combine masks and filter
    valid_apps = merged[time_mask & quirk_mask]

    # 6. Group by deal_id to get the final count
    target_counts = valid_apps.groupby('deal_id').size().reset_index(name=f'apps_after_{n}_days')

    # 7. Merge back to the original deals dataframe (Left merge to keep deals with 0 apps)
    df_deals_updated = df_deals.merge(target_counts, on='deal_id', how='left')
    
    # Fill NaN values with 0 (deals that received no valid applications in the window)
    df_deals_updated[f'apps_after_{n}_days'] = df_deals_updated[f'apps_after_{n}_days'].fillna(0).astype(int)
    
    return df_deals_updated

# Execution:
# Ensure datetimes are set first:
# df_deals['live_since'] = pd.to_datetime(df_deals['live_since'])
# df_apps['application_created_at'] = pd.to_datetime(df_apps['application_created_at'])

df_deals = calculate_target_vectorized(df_deals, df_apps, n=7)


# Location computations tests

In [20]:
import numpy as np

def calculate_haversine_vectorized(lat1, lon1, lat2, lon2):
    """
    Calculates the great-circle distance between two points 
    on the Earth surface in kilometers using NumPy arrays.
    """
    # Earth radius in kilometers
    R = 6371.0 
    
    # Convert degrees to radians (NumPy trig functions require radians)
    lat1_rad = np.radians(lat1)
    lon1_rad = np.radians(lon1)
    lat2_rad = np.radians(lat2)
    lon2_rad = np.radians(lon2)
    
    # Calculate differences
    dlat = lat2_rad - lat1_rad
    dlon = lon2_rad - lon1_rad
    
    # Apply the Haversine formula
    a = np.sin(dlat / 2.0)**2 + np.cos(lat1_rad) * np.cos(lat2_rad) * np.sin(dlon / 2.0)**2
    
    # arcsin is mathematically equivalent to arctan2 for this and often slightly faster in numpy
    c = 2 * np.arcsin(np.sqrt(a)) 
    
    # Return distance in kilometers
    return R * c

1. Create spatial grid: distance between unique company locations and creator locations
2. Filter the creators that are within an eligible range (e.g., 25km)
    - Maybe explore whether I can come up with a metric that is a function of distance, e.g. weighted by distance from deal location
3. Per deal:
    - Total active (eligible) creators in last 7 days/30 days
        - Consider Pro creators? (creators with high follower count/ good reviews)


1. The Pre-Computed Spatial Grid (Calculate Once)

Do not calculate distances between deals and creators. Calculate distances between Unique Deal Locations and Unique Creator Locations.

Locations don't move. A coordinate in Amsterdam is always the same distance from a coordinate in Utrecht.

In [ ]:
# Filter for physical deals with valid locations
df_locs = df_apps[~df_apps['company_location_id'].isna() & (df_apps['deal_type_deals'] == 'physical')]

# 1. Unique Creators
# Grouped by creator_id, grabbing their locations
df_unique_creators = df_locs[['creator_id', 'latitude_creators', 'longitude_creators']].drop_duplicates(subset=['creator_id'])

# 2. Unique Deals/Company Locations 
# Grouped by deal_id (to easily merge back to df_deals later)
df_unique_deals = df_locs[['deal_id', 'latitude_company_locations', 'longitude_company_locations']].drop_duplicates(subset=['deal_id'])

In [39]:
df_locs.deal_id.nunique()

1805

In [40]:
df_locs.company_location_id.nunique()

1014

In [42]:
df_apps.country_creators

0             Belgium
1         Netherlands
2             Belgium
3         Netherlands
4                <NA>
             ...     
269729    Netherlands
269730    Netherlands
269731    Netherlands
269732    Netherlands
269733    Netherlands
Name: country_creators, Length: 269734, dtype: string

In [ ]:
# 1. Unique Company Locations (1014 rows)
df_unique_locations = df_locs[[
    'company_location_id', 
    'latitude_company_locations', 
    'longitude_company_locations',
]].drop_duplicates(subset=['company_location_id'])

# 2. Unique Creators (6696 rows)
df_unique_creators = df_locs[[
    'creator_id', 
    'latitude_creators', 
    'longitude_creators',
    'country_creators'
]].drop_duplicates(subset=['creator_id'])

# 3. The Leaner Cross-Join (~6.79 million rows)
df_spatial_grid = df_unique_locations.merge(df_unique_creators, how='cross')

# 4. Calculate Distances Vectorially
df_spatial_grid['distance_km'] = calculate_haversine_vectorized(
    df_spatial_grid['latitude_company_locations'].values,
    df_spatial_grid['longitude_company_locations'].values,
    df_spatial_grid['latitude_creators'].values,
    df_spatial_grid['longitude_creators'].values
)

# 5. Immediately Filter by Radius (e.g., 50km) and drop coordinates
radius_km = 50.0
df_spatial_grid = df_spatial_grid[df_spatial_grid['distance_km'] <= radius_km]
df_spatial_grid = df_spatial_grid[['company_location_id', 'creator_id', 'distance_km']]

# 6. Map the Spatial Matches Back to the Deals
# We merge the pruned spatial grid with your deals metadata to re-attach the deal_id 
# and the highly crucial created_at timestamp needed for the next step.
df_deals_spatial = df_deals[['deal_id', 'company_location_id', 'created_at']].merge(
    df_spatial_grid, 
    on='company_location_id', 
    how='inner' # Inner join ensures we only keep deals that have creators within the radius
)

In [ ]:
df_deals_meta = df_deals[['deal_id', 'company_location_id', 'created_at', 'accepted_countries']]

df_deals_spatial = df_deals_meta.merge(
    df_spatial_grid, 
    on='company_location_id', 
    how='inner' 
)

# 3. Apply the Country Eligibility Mask
# Assuming accepted_countries_deals is a string like "NL, BE" or a list.
# If it's a string, we can use a vectorized string contains. 
# If it's a list, a quick list comprehension or apply works fine on this reduced dataset.

def is_eligible(row):
    # Handle NaN or missing accepted countries (assuming NaN means all are accepted)
    if pd.isna(row['accepted_countries_deals']):
        return True
    return row['country_creators'] in row['accepted_countries_deals']

df_deals_spatial['is_eligible'] = df_deals_spatial.apply(is_eligible, axis=1)

# Keep only the spatially and legally eligible pairs!
df_deals_spatial = df_deals_spatial[df_deals_spatial['is_eligible']]
df_deals_spatial = df_deals_spatial.drop(columns=['is_eligible'])

In [58]:
df_apps.country_creators.unique()

<StringArray>
[       'Belgium',    'Netherlands',             <NA>,        'Germany',
          'Spain', 'United Kingdom',         'Greece',         'Sweden',
          'Malta',         'France',       'Portugal',  'United States',
          'Italy',         'Poland',        'Denmark',        'Ukraine']
Length: 16, dtype: string

In [67]:
df_deals['accepted_countries']

0            [NLD]
1            [NLD]
2               []
3            [NLD]
4            [NLD]
           ...    
6423         [NLD]
6424         [NLD]
6425    [NLD, BEL]
6426         [BEL]
6427         [NLD]
Name: accepted_countries, Length: 6428, dtype: object

In [44]:
df_spatial_grid

,company_location_id,creator_id,distance_km
1,019bb11f-9d18-015f-1f36-27c82da88c99,7787,1.803499
3,019bb11f-9d18-015f-1f36-27c82da88c99,358,1.803499
6,019bb11f-9d18-015f-1f36-27c82da88c99,9202,1.803499
7,019bb11f-9d18-015f-1f36-27c82da88c99,65,1.803499
8,019bb11f-9d18-015f-1f36-27c82da88c99,220,1.803499
...,...,...,...
4420004,019bb11f-562a-015f-ab80-edf966889f68,8729,17.110075
4420009,019bb11f-562a-015f-ab80-edf966889f68,37862,42.036801
4420011,019bb11f-562a-015f-ab80-edf966889f68,37885,2.622987
4420016,019bb11f-562a-015f-ab80-edf966889f68,38034,2.622987


In [ ]:
# Calculate the distance for all 1.3 million combinations instantly
df_spatial_grid['distance_km'] = calculate_haversine_vectorized(
    df_spatial_grid['deal_lat'], 
    df_spatial_grid['deal_lon'],
    df_spatial_grid['creator_lat'], 
    df_spatial_grid['creator_lon']
)

# Optional: If you only care about creators within a specific radius (e.g., 50km), 
# drop the rest right now to save RAM before you join your time-series data!
df_spatial_grid = df_spatial_grid[df_spatial_grid['distance_km'] <= 50.0]

2. Dynamic Temporal Filtering (Calculate Often)

Now deal with the time aspect. You want to know who was active last week.§§

In [ ]:
# Filter your main creators dataframe (the one with 180k rows)
active_last_week = df_creators[df_creators['last_active'] >= '2026-02-12']

# Count how many ACTIVE creators are sitting at each unique location ID
# Result: A tiny dataframe of 773 rows showing available supply right now
supply_by_location = active_last_week.groupby('creator_loc_id').size().reset_index(name='active_creators')

3. The Final Join (Milliseconds)

Now, map that active supply onto your permanent spatial grid, and filter for deals that have creators within your desired radius (e.g., 25km).

In [ ]:
# Join the active counts to the spatial grid
eligible_supply = df_spatial_grid.merge(supply_by_location, on='creator_loc_id', how='left')

# Drop locations where no one is active
eligible_supply = eligible_supply.dropna(subset=['active_creators'])

# Filter for the radius you care about (e.g., within 25km)
deals_with_supply = eligible_supply[eligible_supply['distance_km'] <= 25]

# Group by deal to see total eligible creators nearby!
final_deal_supply = deals_with_supply.groupby('deal_loc_id')['active_creators'].sum()

In [6]:
df_locs = df_apps[~df_apps['company_location_id'].isna()]

In [11]:
creator_locs = df_locs[['longitude_creator', 'latitude_creator']]
partner_locs = df_locs[['longitude_partner', 'latitude_partner']]